# Mini Project 1 · Group A
## Detection and Analysis of Performance Loss in a PLC-Controlled Production System Using Unsupervised Learning
### Stage 1: Data Ingestion, Integrity Validation, and 85 ms Grid Synchronization

**Lead Author:** Neeraj Bhuma  
**Team Members:** Neeraj Bhuma, Lakshmi Narayana Reddy Mandi, Naga Sai Priyanka Truvidhula  
**Supervisor:** Prof. Felix Georg Müller  
**Institution:** Hochschule Albstadt-Sigmaringen (November 2026)

In [1]:
import pandas as pd
import os
from pathlib import Path

# Dynamically locate project root, raw data folder, and output folder
def get_project_dirs():
    current = Path.cwd().resolve()
    root = next((p for p in [current] + list(current.parents) if (p / "02_Data_and_Outputs").exists() or (p / "01_Notebooks").exists()), current)
    raw_candidates = [
        root / "02_Data_and_Outputs" / "Raw_Data",
        root / "Raw_Data",
        current / "Raw_Data",
        current.parent / "02_Data_and_Outputs" / "Raw_Data",
        current.parent / "Raw_Data",
    ]
    raw_dir = next((p for p in raw_candidates if p.exists() and (p / "Beispieldatensatz Beckhoff SPS.csv").exists()), root / "02_Data_and_Outputs" / "Raw_Data")
    out_candidates = [
        root / "02_Data_and_Outputs" / "Generated_Outputs",
        root / "Generated_Outputs",
        current / "Generated_Outputs",
        current.parent / "02_Data_and_Outputs" / "Generated_Outputs",
    ]
    out_dir = next((p for p in out_candidates if p.exists()), root / "02_Data_and_Outputs" / "Generated_Outputs")
    out_dir.mkdir(parents=True, exist_ok=True)
    return root, raw_dir, out_dir

PROJECT_ROOT, DATA_DIR, OUT_DIR = get_project_dirs()
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")

# Read both files as text before converting their columns.
beck = pd.read_csv(
    DATA_DIR / "Beispieldatensatz Beckhoff SPS.csv",
    sep=";",      # columns are separated by semicolons
    dtype=str,    # read every column as text
)
sie = pd.read_csv(
    DATA_DIR / "Beispieldatensatz Siemens S7 SPS.csv",
    sep=";",      # columns are separated by semicolons
    dtype=str,    # read every column as text
)

print("Beckhoff:", beck.shape, "| Siemens:", sie.shape)  # (rows, columns)


Project root: /Users/lakshminarayanareddymandu/Downloads/B4 Project
Data directory: /Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Raw_Data
Output directory: /Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Generated_Outputs
Beckhoff: (5197, 55) | Siemens: (6283, 144)


**What**
- Load the two original PLC logs, the Beckhoff and the Siemens CSV, from the Example batch-Dataset folder

**Why**
- `dtype=str`, so pandas cannot guess types, change values or turn empty cells into something else before we check them
- The cell loads its own inputs, so it runs in a fresh notebook

**Output**
- Beckhoff: (5197, 55) | Siemens: (6283, 144)

**Next**
- Convert the text timestamps to real UTC times and check that none are empty

In [2]:
# Report empty timestamps without removing their rows.
print("Beckhoff empty timestamps:", (beck["timestamp"] == "").sum())
print("Siemens empty timestamps:", (sie["timestamp"] == "").sum())

# Add UTC time without changing the original timestamp column.
beck["utc_time"] = pd.to_datetime(
    pd.to_numeric(beck["timestamp"]),  # text -> number
    unit="ms",                         # Unix time in milliseconds
    utc=True,                          # timezone-aware UTC
)
sie["utc_time"] = pd.to_datetime(
    pd.to_numeric(sie["timestamp"]),   # text -> number
    unit="ms",                         # Unix time in milliseconds
    utc=True,                          # timezone-aware UTC
)

print("Beckhoff rows and columns:", beck.shape)
print("Siemens rows and columns:", sie.shape)

print(beck[["timestamp", "utc_time"]].head())
print(sie[["timestamp", "utc_time"]].head())

Beckhoff empty timestamps: 0
Siemens empty timestamps: 0
Beckhoff rows and columns: (5197, 56)
Siemens rows and columns: (6283, 145)
       timestamp                      utc_time
0  1569522074925 2019-09-26 18:21:14.925+00:00
1  1569522075030 2019-09-26 18:21:15.030+00:00
2  1569522075128 2019-09-26 18:21:15.128+00:00
3  1569522075231 2019-09-26 18:21:15.231+00:00
4  1569522075330 2019-09-26 18:21:15.330+00:00
       timestamp                      utc_time
0  1569522074515 2019-09-26 18:21:14.515+00:00
1  1569522074773 2019-09-26 18:21:14.773+00:00
2  1569522074850 2019-09-26 18:21:14.850+00:00
3  1569522074972 2019-09-26 18:21:14.972+00:00
4  1569522075064 2019-09-26 18:21:15.064+00:00


**What**
- Load the two original PLC logs, the Beckhoff and the Siemens CSV, from the Example batch-Dataset folder

**Why**
- `dtype=str`, so pandas cannot guess types, change values or turn empty cells into something else before we check them
- The cell loads its own inputs, so it runs in a fresh notebook

**Output**
- Beckhoff: (5197, 55) | Siemens: (6283, 144)

**Next**
- Convert the text timestamps to real UTC times and check that none are empty

In [3]:
# Make separate numeric timestamps for calculations only.
beck_ts = pd.to_numeric(beck["timestamp"])
sie_ts = pd.to_numeric(sie["timestamp"])

# Difference between each timestamp and the previous one.
beck_gaps = beck_ts.diff()
sie_gaps = sie_ts.diff()

# Summarize both files side by side.
gap_stats = pd.DataFrame({
    "Beckhoff": beck_gaps.describe().round(2),  # count, mean, std, ...
    "Siemens": sie_gaps.describe().round(2)
})

gap_stats.insert(0, "Unit", [
    "gaps", "ms", "ms", "ms", "ms", "ms", "ms", "ms"
])

gap_stats.insert(0, "Meaning", [
    "Number of valid timestamp differences",
    "Mean: average gap",
    "Standard deviation: spread around the mean",
    "Smallest gap",
    "25th percentile: lower-quarter gap value",
    "Median: middle gap value (50th percentile)",
    "75th percentile: upper-quarter gap value",
    "Largest gap"
])

gap_stats.index.name = "Statistic"
display(gap_stats)


,Meaning,Unit,Beckhoff,Siemens
Statistic,,,,
count,Number of valid timestamp differences,gaps,5196.00,6282.00
mean,Mean: average gap,ms,97.27,80.50
std,Standard deviation: spread around the mean,ms,33.78,44.24
min,Smallest gap,ms,78.00,26.00
25%,25th percentile: lower-quarter gap value,ms,93.00,74.00
50%,Median: middle gap value (50th percentile),ms,93.00,76.00
75%,75th percentile: upper-quarter gap value,ms,97.00,78.00
max,Largest gap,ms,1841.00,1722.00


**What**
- Calculated the time between each row and the previous row, in milliseconds, for both PLCs
- Put the summary of both in one table, with a plain-language meaning for each statistic

**Why**
- The two logs do not record at a fixed rate, and the sampling interval decides which grid we can use later
- We use separate numeric series for the calculation, so the text columns stay unchanged

**Output**
- Beckhoff: 5196 gaps, mean 97.27 ms, minimum 78 ms
- Siemens: 6282 gaps, mean 80.50 ms, minimum 26 ms
- Beckhoff is slower and more regular, Siemens is faster and less regular

**Next**
- Plot the recorded times of both files

In [4]:
import plotly.graph_objects as go

# Show each log on its own row, sharing the same time axis.
fig = go.Figure()

fig.add_scatter(
    x=beck["utc_time"],
    y=["Beckhoff"] * len(beck),  # one row for this PLC
    mode="markers",              # one dot per record
    name="Beckhoff",
)
fig.add_scatter(
    x=sie["utc_time"],
    y=["Siemens"] * len(sie),    # one row for this PLC
    mode="markers",              # one dot per record
    name="Siemens",
)

fig.update_layout(
    title="Recorded timestamps of both logs",
    xaxis_title="UTC time",
    yaxis_title="PLC log",
    xaxis_rangeslider_visible=True,       # slider to zoom in time
    xaxis_tickformat="%H:%M:%S.%L",       # show milliseconds
    xaxis_hoverformat="%H:%M:%S.%L",
)

fig.show()

**What**
- Plot every recorded timestamp of both logs on one shared UTC time axis, Beckhoff on one row and Siemens on the other

**Why**
- A picture shows at once whether both logs cover the same period and whether there are holes
- The range slider lets us zoom into short stretches

**Output**
- Interactive plotly chart, no printed text
- Two almost solid bands over the same period, the Siemens band is denser

**Next**
- Read the exact start and end of both logs and their overlap

In [5]:
# Find the earliest and latest UTC timestamps in each file.
beck_start = beck["utc_time"].min()
beck_end = beck["utc_time"].max()
sie_start = sie["utc_time"].min()
sie_end = sie["utc_time"].max()

coverage = pd.DataFrame({
    "Beckhoff": [len(beck), beck_start, beck_end],
    "Siemens": [len(sie), sie_start, sie_end],
}, index=[
    "Number of rows",
    "Earliest timestamp (UTC)",
    "Latest timestamp (UTC)",
])
display(coverage)

# Calculate the overlap of the two recorded time ranges.
shared_start = max(beck_start, sie_start)  # the later start
shared_end = min(beck_end, sie_end)        # the earlier end

if shared_start <= shared_end:
    print("Overlap starts:", shared_start)
    print("Overlap ends:", shared_end)
    print(
        "Overlap duration (seconds):",
        (shared_end - shared_start).total_seconds(),
    )
else:
    print("The recorded time ranges do not overlap.")

,Beckhoff,Siemens
Number of rows,5197,6283
Earliest timestamp (UTC),2019-09-26 18:21:14.925000+00:00,2019-09-26 18:21:14.515000+00:00
Latest timestamp (UTC),2019-09-26 18:29:40.319000+00:00,2019-09-26 18:29:40.239000+00:00


Overlap starts: 2019-09-26 18:21:14.925000+00:00
Overlap ends: 2019-09-26 18:29:40.239000+00:00
Overlap duration (seconds): 505.314


**What**
- Find the earliest and latest UTC time in each file
- Calculate how long both logs overlap

**Why**
- We only trust the merged data where both PLCs were recording
- We want to know how much of each log lies outside that window

**Output**
- Beckhoff: 18:21:14.925 to 18:29:40.319, Siemens: 18:21:14.515 to 18:29:40.239 (2019-09-26, UTC)
- Overlap from 18:21:14.925 to 18:29:40.239, duration 505.314 s (about 8.4 minutes)

**Next**
- Check for repeated and out-of-order timestamps


In [6]:
print(
    "Beckhoff repeated timestamps:",
    beck["timestamp"].duplicated().sum()
)
print(
    "Siemens repeated timestamps:",
    sie["timestamp"].duplicated().sum()
)


Beckhoff repeated timestamps: 0
Siemens repeated timestamps: 0


In [7]:
checks = pd.DataFrame({
    "Beckhoff": [
        beck["utc_time"].isna().sum(),                # invalid times
        beck["utc_time"].is_monotonic_increasing,     # sorted?
        beck["utc_time"].duplicated().sum(),          # repeats
    ],
    "Siemens": [
        sie["utc_time"].isna().sum(),                 # invalid times
        sie["utc_time"].is_monotonic_increasing,      # sorted?
        sie["utc_time"].duplicated().sum(),           # repeats
    ],
}, index=[
    "Rows without a valid UTC time",
    "Timestamps in increasing order",
    "Repeated timestamps",
])

display(checks)

,Beckhoff,Siemens
Rows without a valid UTC time,0,0
Timestamps in increasing order,True,True
Repeated timestamps,0,0


**What**
- Collect three quality checks per PLC in one table: rows without a valid UTC time, whether the times are in increasing order, and how many times are repeated

**Why**
- `merge_asof` in the next cells only works correctly on sorted times, so we check this ourselves instead of assuming it

**Output**
- 0 rows without a valid UTC time, `True` for increasing order, 0 repeated timestamps, for Beckhoff and for Siemens
- So no sorting or cleaning is needed

**Next**
- Build the common 85 ms time grid


In [8]:
# Cover the earliest start through the latest end of both logs.
grid_start = min(beck["utc_time"].min(), sie["utc_time"].min())
grid_end = max(beck["utc_time"].max(), sie["utc_time"].max())

grid = pd.DataFrame({
    "utc_time": pd.date_range(grid_start, grid_end, freq="85ms")  # 85 ms step
})

# Keep the timestamp of each selected source record.
beck_match = beck.assign(beck_time=beck["utc_time"])
sie_match = sie.assign(sie_time=sie["utc_time"])

# Select the latest record at or before each grid time.
grid = pd.merge_asof(
    grid,
    beck_match,
    on="utc_time",
    direction="backward",  # latest record at or before the grid time
)
grid = pd.merge_asof(
    grid,
    sie_match,
    on="utc_time",
    direction="backward",  # latest record at or before the grid time
    suffixes=("_beckhoff", "_siemens"),  # names of repeated columns
)

# Measure how old each selected reading is.
grid["beck_age_ms"] = (
    (grid["utc_time"] - grid["beck_time"]).dt.total_seconds() * 1000
)
grid["sie_age_ms"] = (
    (grid["utc_time"] - grid["sie_time"]).dt.total_seconds() * 1000
)

print("Trial grid rows:", len(grid))
print("Interval: 85 ms")
print(
    "Tail beyond the last grid point (ms):",
    (grid_end - grid["utc_time"].max()).total_seconds() * 1000,
)

Trial grid rows: 5951
Interval: 85 ms
Tail beyond the last grid point (ms): 54.0


**What**
- Build a trial grid with one row every 85 ms median of 76 and 93 ms, from the earliest start to the latest end of both logs
- For each grid time, take the latest record of each PLC at or before that time (`merge_asof`, backward)
- Store how old each selected record is, in milliseconds

**Why**
- The two PLCs record at different, irregular times, so we need one common time base
- Looking backward means we never use a value from the future
- We keep the original source times (`beck_time`, `sie_time`), so every grid row can be traced back to its records

**Output**
- Trial grid rows: 5951
- Interval: 85 ms
- Tail beyond the last grid point: 54.0 ms
- The first grid rows have no Beckhoff record yet, because Siemens starts earlier (18:21:14.515 against 18:21:14.925)

**Next**
- Look at the first rows to see how records repeat


In [9]:
comparison = grid[[
    "utc_time",
    "beck_time",
    "beck_age_ms",
    "sie_time",
    "sie_age_ms",
]].copy()  # copy, so the grid stays unchanged

# Check whether consecutive grid points use the same source record.
comparison["beckhoff_repeated"] = (
    comparison["beck_time"] == comparison["beck_time"].shift()  # row above
)
comparison["siemens_repeated"] = (
    comparison["sie_time"] == comparison["sie_time"].shift()    # row above
)

print(comparison.head(20).to_string(index=False))

                     utc_time                     beck_time  beck_age_ms                      sie_time  sie_age_ms  beckhoff_repeated  siemens_repeated
2019-09-26 18:21:14.515+00:00                           NaT          NaN 2019-09-26 18:21:14.515+00:00         0.0              False             False
2019-09-26 18:21:14.600+00:00                           NaT          NaN 2019-09-26 18:21:14.515+00:00        85.0              False              True
2019-09-26 18:21:14.685+00:00                           NaT          NaN 2019-09-26 18:21:14.515+00:00       170.0              False              True
2019-09-26 18:21:14.770+00:00                           NaT          NaN 2019-09-26 18:21:14.515+00:00       255.0              False              True
2019-09-26 18:21:14.855+00:00                           NaT          NaN 2019-09-26 18:21:14.850+00:00         5.0              False             False
2019-09-26 18:21:14.940+00:00 2019-09-26 18:21:14.925+00:00         15.0 2019-09-26 18:2

**What**
- Take the timing columns of the grid and mark whether a grid row uses the same source record as the row before it, for Beckhoff and for Siemens
- Print the first 20 rows

**Why**
- It shows directly what the 85 ms grid does to each log: where a PLC records slower than 85 ms a record is repeated, where it records faster a record can be skipped

**Output**
- First rows: `beck_time` is NaT and `beck_age_ms` is NaN (no Beckhoff record yet)
- `sie_age_ms` starts at 0.0, 85.0, ... and `siemens_repeated` is True where no newer Siemens record arrived

**Next**
- Count how many original records are never selected

In [10]:
record_coverage = pd.DataFrame({
    "Beckhoff": [
        len(beck),                                # all records
        grid["beck_time"].nunique(),              # selected at least once
        len(beck) - grid["beck_time"].nunique(),  # never selected
    ],
    "Siemens": [
        len(sie),
        grid["sie_time"].nunique(),
        len(sie) - grid["sie_time"].nunique(),
    ],
}, index=[
    "Original records",
    "Records selected at least once",
    "Records never selected",
])

record_coverage.loc["Represented (%)"] = (
    record_coverage.loc["Records selected at least once"]
    / record_coverage.loc["Original records"] * 100
).round(2)

record_coverage.loc["Not selected (%)"] = (
    record_coverage.loc["Records never selected"]
    / record_coverage.loc["Original records"] * 100
).round(2)

display(record_coverage)

/var/folders/6n/0vwd_l_142v_4sjxs08hcr_00000gn/T/ipykernel_1593/2926642486.py:18: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype int64 cannot hold the value being set, so the result has dtype float64. Cast the object to float64 before this operation to retain the current behavior.
  record_coverage.loc["Represented (%)"] = (
/var/folders/6n/0vwd_l_142v_4sjxs08hcr_00000gn/T/ipykernel_1593/2926642486.py:18: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype int64 cannot hold the value being set, so the result has dtype float64. Cast the object to float64 before this operation to retain the current behavior.
  record_coverage.loc["Represented (%)"] = (


,Beckhoff,Siemens
Original records,5197.00,6283.00
Records selected at least once,5172.00,5677.00
Records never selected,25.00,606.00
Represented (%),99.52,90.35
Not selected (%),0.48,9.65


**What**
- Count how many original records the 85 ms grid selects at least once and how many it never selects, for both PLCs

**Why**
- A record that is never selected is lost in the merged table

**Output**
- Beckhoff: 5172 of 5197 selected, 25 never selected (0.48%)
- Siemens: 5677 of 6283 selected, 606 never selected (9.65%)

**Next**
- Compare with a 93 ms grid (cell 11)

In [11]:
out_path = OUT_DIR / "merged_85ms.csv"
grid.to_csv(out_path, index=False)  # index=False: no row numbers in file

print("Saved:", out_path)
print("Rows:", len(grid), "| Columns:", len(grid.columns))


Saved: /Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Generated_Outputs/merged_85ms.csv
Rows: 5951 | Columns: 204


**What**
- Save the first merged table as `merged_85ms.csv` in the output directory folder

**Why**
- Keeps a copy of the merged table before cleaning
- The cleaned table in cell 26 overwrites this file

**Output**
- Saved: the path of `merged_85ms.csv`
- Rows: 5951 | Columns: 204

**Next**
- Look at the merged timing columns (cell 13), then at the column names

In [12]:
print("Merged rows:", len(grid))
print("Merged columns:", len(grid.columns))

timing_columns = [
    "utc_time",
    "timestamp_beckhoff",  # original Beckhoff timestamp
    "beck_time",           # time of the selected Beckhoff record
    "beck_age_ms",         # age of that record at the grid time
    "timestamp_siemens",   # original Siemens timestamp
    "sie_time",            # time of the selected Siemens record
    "sie_age_ms",          # age of that record at the grid time
]

display(grid[timing_columns].head(10))

Merged rows: 5951
Merged columns: 204


,utc_time,timestamp_beckhoff,beck_time,beck_age_ms,timestamp_siemens,sie_time,sie_age_ms
0,2019-09-26 18:21:14.515+00:00,NaN,NaT,NaN,1569522074515,2019-09-26 18:21:14.515+00:00,0.0
1,2019-09-26 18:21:14.600+00:00,NaN,NaT,NaN,1569522074515,2019-09-26 18:21:14.515+00:00,85.0
2,2019-09-26 18:21:14.685+00:00,NaN,NaT,NaN,1569522074515,2019-09-26 18:21:14.515+00:00,170.0
3,2019-09-26 18:21:14.770+00:00,NaN,NaT,NaN,1569522074515,2019-09-26 18:21:14.515+00:00,255.0
4,2019-09-26 18:21:14.855+00:00,NaN,NaT,NaN,1569522074850,2019-09-26 18:21:14.850+00:00,5.0
5,2019-09-26 18:21:14.940+00:00,1569522074925,2019-09-26 18:21:14.925+00:00,15.0,1569522074850,2019-09-26 18:21:14.850+00:00,90.0
6,2019-09-26 18:21:15.025+00:00,1569522074925,2019-09-26 18:21:14.925+00:00,100.0,1569522074972,2019-09-26 18:21:14.972+00:00,53.0
7,2019-09-26 18:21:15.110+00:00,1569522075030,2019-09-26 18:21:15.030+00:00,80.0,1569522075064,2019-09-26 18:21:15.064+00:00,46.0
8,2019-09-26 18:21:15.195+00:00,1569522075128,2019-09-26 18:21:15.128+00:00,67.0,1569522075152,2019-09-26 18:21:15.152+00:00,43.0
9,2019-09-26 18:21:15.280+00:00,1569522075231,2019-09-26 18:21:15.231+00:00,49.0,1569522075240,2019-09-26 18:21:15.240+00:00,40.0


**What**
- Print the number of rows and columns of the merged table
- Show the first 10 rows of the timing columns

**Why**
- A quick check that the merge kept all grid rows and added the source times and ages

**Output**
- Merged rows: 5951
- Merged columns: 204
- First rows: `timestamp_beckhoff` is NaN at first, then 1569522074925 from 18:21:14.940

**Next**
- List all original columns and what they become in the merged table

In [13]:
rows = []

for source, table in (("Beckhoff", beck), ("Siemens", sie)):
    for name in table.columns:
        # Timing columns were already explained separately.
        if name in ("timestamp", "utc_time"):
            continue

        # Names present in both files get a PLC suffix after the merge.
        merged_name = name
        if name in beck.columns and name in sie.columns:
            merged_name = name + "_" + source.lower()

        rows.append({
            "Source": source,                # which PLC
            "Original column": name,         # name in the CSV
            "Merged column": merged_name,    # name in the merged table
        })

inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))

  Source                        Original column                          Merged column
Beckhoff              VG.I1_ref_switch_vertical              VG.I1_ref_switch_vertical
Beckhoff            VG.I2_ref_switch_horizontal            VG.I2_ref_switch_horizontal
Beckhoff                VG.I3_ref_switch_rotate                VG.I3_ref_switch_rotate
Beckhoff        VG.B1_encoder_vertical_impulse1        VG.B1_encoder_vertical_impulse1
Beckhoff        VG.B2_encoder_vertical_impulse2        VG.B2_encoder_vertical_impulse2
Beckhoff      VG.B3_encoder_horizontal_impulse1      VG.B3_encoder_horizontal_impulse1
Beckhoff      VG.B4_encoder_horizontal_impulse2      VG.B4_encoder_horizontal_impulse2
Beckhoff          VG.B5_encoder_rotate_impulse1          VG.B5_encoder_rotate_impulse1
Beckhoff          VG.B6_encoder_rotate_impulse2          VG.B6_encoder_rotate_impulse2
Beckhoff            WH.I1_Ref_switch_horizontal            WH.I1_Ref_switch_horizontal
Beckhoff             WH.I2_Light_barrier_in

**What**
- Build an inventory of every original column of both PLC tables, without the two time columns
- For each column: source PLC, original name, name in the merged table

**Why**
- We need an exact list of what we have before we look up what each signal means
- A name that exists in both files gets the suffix `_beckhoff` or `_siemens`, so we do not guess the merged names

**Output**
- Printed table with 197 entries (54 from Beckhoff, 143 from Siemens)
- Beckhoff entries start with names like `VG.I1_ref_switch_vertical`
- Two entries are called `Unnamed: 54` and `Unnamed: 143`: pandas names for columns with an empty header

**Next**
- Find the signal dictionary file and read it

In [14]:
import glob

# Needed once so that pandas can read .xlsx files (Jupyter command).
%pip install openpyxl

# Find the dictionary without guessing its unusual filename spelling.
matches = sorted(glob.glob(os.path.join(str(DATA_DIR), "Variablen*.xlsx")))
if not matches:
    matches = sorted(glob.glob(os.path.join(str(PROJECT_ROOT), "**/Variablen*.xlsx"), recursive=True))
print("Dictionary files found:", matches)

if len(matches) >= 1:  # continue only if at least one file matches
    with pd.ExcelFile(matches[0]) as dictionary:
        print("Sheets:", dictionary.sheet_names)

        # Preview the first sheet before deciding how to use it.
        dictionary_preview = pd.read_excel(dictionary, sheet_name=0)
        print(dictionary_preview.head(10).to_string(index=False))
else:
    print("Expected one dictionary file. Check the matches above.")


Note: you may need to restart the kernel to use updated packages.
Dictionary files found: ['/Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Raw_Data/Variablenerkl_rung.xlsx']


Sheets: ['Siemens', 'Beckhoff']
Variablenname Datentyp                                        Kommentar          Station                   Symbol
         A0.0     Bool                                         Motor=on  Sorting station  M_long_Conveyor_belt_on
         A0.1     Bool           sorting station compressor on (output)  Sorting station    compressor_on_sorting
         A0.2     Bool White Block, Pneumatic pusher activated (output)  Sorting station        pneu_pusher_white
         A0.3     Bool   Red Block, Pneumatic pusher activated (output)  Sorting station          pneu_pusher_red
         A0.4     Bool  Blue block, Pneumatic pusher activated (output)  Sorting station         pneu_pusher_blue
         A0.5     Bool                          cw = clockwise (output) Drilling station       motor_turntable_cw
         A0.6     Bool                  ccw = counterclockwise (output) Drilling station      motor_turntable_ccw
         A0.7     Bool                                  

**What**
- Install `openpyxl` (pandas needs it to read `.xlsx`)
- Find the dictionary file with the pattern `Variablen*.xlsx`, list its sheets and preview the first sheet

**Why**
- The file name has an unusual spelling, so we search for it instead of typing it
- We only continue if exactly one file matches

**Output**
- One dictionary file found
- Sheets: ['Siemens', 'Beckhoff']
- Ten rows of the first (Siemens) sheet with the columns Variablenname, Datentyp, Kommentar, Station, Symbol
- The pip lines say "Requirement already satisfied"

**Next**
- Check the dictionary columns
- Note: `matches` is needed by cells 20 and 21. If cells 16 to 19 are dropped, the preview lines (from `dictionary_preview` on) can be deleted

In [15]:
print("Dictionary column headers:")
print(dictionary_preview.columns.tolist())  # header names as a list

Dictionary column headers:
['Variablenname', 'Datentyp', 'Kommentar', 'Station', 'Symbol']


In [16]:
# Check for ambiguous dictionary identifiers.
name_counts = dictionary_preview["Variablenname"].value_counts()
symbol_counts = dictionary_preview["Symbol"].value_counts()

print("Repeated variable names:", (name_counts > 1).sum())  # should be 0
print("Repeated symbols:", (symbol_counts > 1).sum())

# Compare original column names with both dictionary fields.
by_name = inventory["Original column"].isin(
    dictionary_preview["Variablenname"]
)
by_symbol = inventory["Original column"].isin(
    dictionary_preview["Symbol"]
)

print("Columns matching variable names:", by_name.sum())
print("Columns matching symbols:", by_symbol.sum())
print("Columns matching neither:", (~by_name & ~by_symbol).sum())


Repeated variable names: 0
Repeated symbols: 1
Columns matching variable names: 142
Columns matching symbols: 0
Columns matching neither: 55


**What**
- Check whether variable names and symbols in the dictionary are unique
- Compare our original column names with both dictionary fields

**Why**
- A repeated identifier would make the lookup ambiguous

**Output**
- Repeated variable names: 0
- Repeated symbols: 1
- Columns matching variable names: 142
- Columns matching symbols: 0
- Columns matching neither: 55 (only the Siemens sheet was read, see cell 21)

**Next**
- List the 55 columns without a match

In [17]:
unmatched = inventory.loc[~by_name & ~by_symbol]  # matches neither field

print("Entries without an exact dictionary match:", len(unmatched))
print(unmatched.to_string(index=False))


Entries without an exact dictionary match: 55
  Source                        Original column                          Merged column
Beckhoff              VG.I1_ref_switch_vertical              VG.I1_ref_switch_vertical
Beckhoff            VG.I2_ref_switch_horizontal            VG.I2_ref_switch_horizontal
Beckhoff                VG.I3_ref_switch_rotate                VG.I3_ref_switch_rotate
Beckhoff        VG.B1_encoder_vertical_impulse1        VG.B1_encoder_vertical_impulse1
Beckhoff        VG.B2_encoder_vertical_impulse2        VG.B2_encoder_vertical_impulse2
Beckhoff      VG.B3_encoder_horizontal_impulse1      VG.B3_encoder_horizontal_impulse1
Beckhoff      VG.B4_encoder_horizontal_impulse2      VG.B4_encoder_horizontal_impulse2
Beckhoff          VG.B5_encoder_rotate_impulse1          VG.B5_encoder_rotate_impulse1
Beckhoff          VG.B6_encoder_rotate_impulse2          VG.B6_encoder_rotate_impulse2
Beckhoff            WH.I1_Ref_switch_horizontal            WH.I1_Ref_switch_horizont

x**What**
- List the inventory entries without an exact dictionary match

**Why**
- We want to see which columns are affected before we change anything

**Output**
- 55 entries, mostly Beckhoff columns such as `VG.I1_ref_switch_vertical`

**Next**
- Read both dictionary sheets properly (cells 20 and 21)

In [18]:

# Match using the variable-name field, which was unique.
described = inventory.merge(
    dictionary_preview,
    how="left",                   # keep every inventory entry
    left_on="Original column",
    right_on="Variablenname",
    validate="m:1",               # error if the lookup is ambiguous
)

print("Original inventory entries:", len(inventory))
print("Description-table entries:", len(described))
print(
    "Entries without a dictionary match:",
    described["Variablenname"].isna().sum(),
)

# Show names, descriptions, stations and data types together.
display(described[[
    "Source",
    "Original column",
    "Merged column",
    "Datentyp",
    "Kommentar",
    "Station",
    "Symbol",
]])


Original inventory entries: 197
Description-table entries: 197
Entries without a dictionary match: 55


,Source,Original column,Merged column,Datentyp,Kommentar,Station,Symbol
0,Beckhoff,VG.I1_ref_switch_vertical,VG.I1_ref_switch_vertical,NaN,NaN,NaN,NaN
1,Beckhoff,VG.I2_ref_switch_horizontal,VG.I2_ref_switch_horizontal,NaN,NaN,NaN,NaN
2,Beckhoff,VG.I3_ref_switch_rotate,VG.I3_ref_switch_rotate,NaN,NaN,NaN,NaN
3,Beckhoff,VG.B1_encoder_vertical_impulse1,VG.B1_encoder_vertical_impulse1,NaN,NaN,NaN,NaN
4,Beckhoff,VG.B2_encoder_vertical_impulse2,VG.B2_encoder_vertical_impulse2,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
192,Siemens,E200.0,E200.0,Bool,Reset button from HMI,Beckhoff,Button_HMI_reset
193,Siemens,E200.1,E200.1,Bool,Not used,-,-
194,Siemens,E1.7,E1.7,Bool,Not used,-,-
195,Siemens,E200.2,E200.2,Bool,Not used,-,-


**What**
- Merge the inventory with the first dictionary sheet using the variable name

**Why**
- Variable names were unique, so the lookup is safe (`validate="m:1"`)

**Output**
- Original inventory entries: 197
- Description-table entries: 197
- Entries without a dictionary match: 55
- Table with data type, comment, station and symbol

**Next**
- Read both sheets and use the right headers of each (cells 20 and 21)

In [19]:
with pd.ExcelFile(matches[0]) as book:
    print("Sheets:", book.sheet_names)

    for sheet in book.sheet_names:
        preview = pd.read_excel(book, sheet_name=sheet)  # one sheet

        print("\nSheet:", sheet)
        print("Columns:", preview.columns.tolist())
        print(preview.head(3).to_string(index=False))
        

Sheets: ['Siemens', 'Beckhoff']

Sheet: Siemens
Columns: ['Variablenname', 'Datentyp', 'Kommentar', 'Station', 'Symbol']
Variablenname Datentyp                                        Kommentar         Station                  Symbol
         A0.0     Bool                                         Motor=on Sorting station M_long_Conveyor_belt_on
         A0.1     Bool           sorting station compressor on (output) Sorting station   compressor_on_sorting
         A0.2     Bool White Block, Pneumatic pusher activated (output) Sorting station       pneu_pusher_white

Sheet: Beckhoff
Columns: ['Variablenname / Beschreibung', 'Station']
Variablenname / Beschreibung       Station
  Pallets.Conveyor_belt_lift Conveyor belt
   VG.I1_ref_switch_vertical         Robot
 VG.I2_ref_switch_horizontal         Robot


**What**
- Open the dictionary again and print, for every sheet, its columns and the first three rows

**Why**
- Cell 15 only showed the first sheet
- We want to see how the Beckhoff sheet is built before we merge, because its headers differ from the Siemens sheet
- Inspection only, nothing depends on it, but it documents why cell 21 reads the sheets differently

**Output**
- Sheets: ['Siemens', 'Beckhoff']
- Siemens: Variablenname, Datentyp, Kommentar, Station, Symbol (for example A0.0 is Motor=on, Sorting station)
- Beckhoff: other headers, among them "Variablenname / Beschreibung" and "Station"

**Next**
- Build one lookup table from both sheets with the right column names

In [20]:
siemens_dict = pd.read_excel(matches[0], sheet_name="Siemens")
beckhoff_dict = pd.read_excel(matches[0], sheet_name="Beckhoff")

# Keep each dictionary entry linked to its source PLC.
siemens_lookup = siemens_dict[[
    "Variablenname", "Datentyp", "Kommentar", "Station", "Symbol",
]].assign(Source="Siemens")

# The Beckhoff sheet only offers a name and a station.
beckhoff_lookup = pd.DataFrame({
    "Variablenname": beckhoff_dict["Variablenname / Beschreibung"],
    "Station": beckhoff_dict["Station"],
    "Source": "Beckhoff",
})

lookup = pd.concat(
    [siemens_lookup, beckhoff_lookup], ignore_index=True
)

# Check uniqueness within each PLC's dictionary.
ambiguous = lookup.duplicated(subset=["Source", "Variablenname"])
print("Repeated identifiers within a sheet:", ambiguous.sum())  # expect 0

if ambiguous.any():
    print("Stop here and paste the output before continuing.")
else:
    described = inventory.merge(
        lookup,
        how="left",                                # keep all 197 entries
        left_on=["Source", "Original column"],
        right_on=["Source", "Variablenname"],
        validate="m:1",                            # error if ambiguous
        indicator=True,                            # adds the _merge column
    )

    print("Inventory entries:", len(inventory))
    print("Description-table entries:", len(described))
    print(
        "Entries with a dictionary match:",
        (described["_merge"] == "both").sum(),
    )

    still_missing = described.loc[
        described["_merge"] == "left_only",        # no dictionary match
        ["Source", "Original column"],
    ]
    print("Still unmatched:", len(still_missing))
    print(still_missing.to_string(index=False))

    display(described[[
        "Source", "Original column", "Merged column",
        "Datentyp", "Kommentar", "Station", "Symbol",
    ]])

Repeated identifiers within a sheet: 0
Inventory entries: 197
Description-table entries: 197
Entries with a dictionary match: 195
Still unmatched: 2
  Source Original column
Beckhoff     Unnamed: 54
 Siemens    Unnamed: 143


,Source,Original column,Merged column,Datentyp,Kommentar,Station,Symbol
0,Beckhoff,VG.I1_ref_switch_vertical,VG.I1_ref_switch_vertical,NaN,NaN,Robot,NaN
1,Beckhoff,VG.I2_ref_switch_horizontal,VG.I2_ref_switch_horizontal,NaN,NaN,Robot,NaN
2,Beckhoff,VG.I3_ref_switch_rotate,VG.I3_ref_switch_rotate,NaN,NaN,Robot,NaN
3,Beckhoff,VG.B1_encoder_vertical_impulse1,VG.B1_encoder_vertical_impulse1,NaN,NaN,Robot,NaN
4,Beckhoff,VG.B2_encoder_vertical_impulse2,VG.B2_encoder_vertical_impulse2,NaN,NaN,Robot,NaN
...,...,...,...,...,...,...,...
192,Siemens,E200.0,E200.0,Bool,Reset button from HMI,Beckhoff,Button_HMI_reset
193,Siemens,E200.1,E200.1,Bool,Not used,-,-
194,Siemens,E1.7,E1.7,Bool,Not used,-,-
195,Siemens,E200.2,E200.2,Bool,Not used,-,-


**What**
- Read both dictionary sheets and put them in one lookup table that keeps the source PLC
- Siemens gives name, data type, comment, station and symbol; Beckhoff gives only name and station
- Check that no identifier is repeated inside one PLC, then merge the lookup with our inventory using (Source, column name)

**Why**
- The same name could exist in both PLCs, so the source must be part of the key
- `validate="m:1"` stops with an error if the lookup is ambiguous
- `indicator=True` tells us which entries found a match
- We do not change any column name to force a match

**Output**
- Repeated identifiers within a sheet: 0
- Inventory entries: 197, description-table entries: 197
- Entries with a dictionary match: 195
- Still unmatched: 2, namely Beckhoff `Unnamed: 54` and Siemens `Unnamed: 143`
- Beckhoff rows have empty data type, comment and symbol, because that sheet does not have them
- So 195 of 197 signals can be named from the dictionary

**Next**
- Look at missing and empty cells in the merged table



In [21]:
# Build a separate mask of missing and empty cells.
missing_mask = grid.replace(
    to_replace=r"^\s*$",   # empty or whitespace-only text
    value=pd.NA,           # treat it as missing
    regex=True,
).isna()                   # True where a cell is missing

# Count affected cells in each column.
missing_counts = missing_mask.sum()
print("Missing or empty cells per column:")
print(missing_counts[missing_counts > 0].to_string())

# Find the rows containing at least one affected cell.
affected_rows = missing_mask.any(axis=1)
print("Affected rows:", affected_rows.sum(), "of", len(grid))

# Show row indices and counts for the first 20 affected rows.
print("Affected row indices and number of empty cells:")
print(missing_mask.sum(axis=1).loc[affected_rows].head(20))

# Inspect their actual values, keeping the original row indices.
display(grid.loc[affected_rows].head(20))

Missing or empty cells per column:
timestamp_beckhoff                           5
VG.I1_ref_switch_vertical                    5
VG.I2_ref_switch_horizontal                  5
VG.I3_ref_switch_rotate                      5
VG.B1_encoder_vertical_impulse1              5
VG.B2_encoder_vertical_impulse2              5
VG.B3_encoder_horizontal_impulse1            5
VG.B4_encoder_horizontal_impulse2            5
VG.B5_encoder_rotate_impulse1                5
VG.B6_encoder_rotate_impulse2                5
WH.I1_Ref_switch_horizontal                  5
WH.I2_Light_barrier_inside                   5
WH.I3_Light_barrier_outside                  5
WH.I4_Ref_swtich_vertical                    5
WH.I5_Ref_switch_cantilever_front            5
WH.I6_Ref_switch_cantilever_back             5
WH.A1_trail_sensor_lower                     5
WH.A2_trail_sensor_upper                     5
WH.B1_encoder_horizontal_impuls1             5
WH.B2_encoder_horizontal_impuls2             5
WH.B3_encoder_vertical_im

Affected rows: 5951 of 5951
Affected row indices and number of empty cells:
0     58
1     58
2     58
3     58
4     58
5      2
6      2
7      2
8      2
9      2
10     2
11     2
12     2
13     2
14     2
15     2
16     2
17     2
18     2
19     2
dtype: int64


,utc_time,timestamp_beckhoff,VG.I1_ref_switch_vertical,VG.I2_ref_switch_horizontal,VG.I3_ref_switch_rotate,VG.B1_encoder_vertical_impulse1,VG.B2_encoder_vertical_impulse2,VG.B3_encoder_horizontal_impulse1,VG.B4_encoder_horizontal_impulse2,VG.B5_encoder_rotate_impulse1,...,E6.0,E6.1,E200.0,E200.1,E1.7,E200.2,Unnamed: 143,sie_time,beck_age_ms,sie_age_ms
0,2019-09-26 18:21:14.515+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.515+00:00,NaN,0.0
1,2019-09-26 18:21:14.600+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.515+00:00,NaN,85.0
2,2019-09-26 18:21:14.685+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.515+00:00,NaN,170.0
3,2019-09-26 18:21:14.770+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.515+00:00,NaN,255.0
4,2019-09-26 18:21:14.855+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.850+00:00,NaN,5.0
5,2019-09-26 18:21:14.940+00:00,1569522074925,True,True,False,True,False,False,False,False,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.850+00:00,15.0,90.0
6,2019-09-26 18:21:15.025+00:00,1569522074925,True,True,False,True,False,False,False,False,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:14.972+00:00,100.0,53.0
7,2019-09-26 18:21:15.110+00:00,1569522075030,True,True,False,True,False,False,False,False,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:15.064+00:00,80.0,46.0
8,2019-09-26 18:21:15.195+00:00,1569522075128,True,True,False,True,False,False,False,False,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:15.152+00:00,67.0,43.0
9,2019-09-26 18:21:15.280+00:00,1569522075231,True,True,False,True,False,False,False,False,...,True,True,False,False,False,False,NaN,2019-09-26 18:21:15.240+00:00,49.0,40.0


**What**
- Build a separate true/false table (`missing_mask`) that marks every missing or empty cell of the merged table
- A cell counts as empty if it is NaN or only whitespace
- Count affected cells per column, count affected rows, show the first 20 of them

**Why**
- Before we decide anything about missing data, we want to see exactly where it is and how much
- The merged table itself is not changed

**Output**
- 5 empty cells in each Beckhoff column: the first 5 grid rows, where Siemens already had data and Beckhoff had not started
- `Unnamed: 54` and `Unnamed: 143` empty in every row
- Number of affected rows and the first 20 rows (read the exact counts from the run)

**Next**
- Remove the two fully empty columns and check again

In [22]:
# Remove only the two fully empty columns from the merged table.
grid = grid.drop(columns=["Unnamed: 54", "Unnamed: 143"])  # by name only

# Check missing and empty cells again.
missing_mask = grid.replace(
    to_replace=r"^\s*$",   # empty or whitespace-only text
    value=pd.NA,
    regex=True,
).isna()

rows_with_gaps = missing_mask.any(axis=1)  # rows with at least one gap

report = pd.DataFrame({
    "utc_time": grid.loc[rows_with_gaps, "utc_time"],
    "beck_time": grid.loc[rows_with_gaps, "beck_time"],
    "sie_time": grid.loc[rows_with_gaps, "sie_time"],
    "empty_cells": missing_mask.sum(axis=1).loc[rows_with_gaps],
})

print("Total rows:", len(grid))
print("Rows still containing empty cells:", rows_with_gaps.sum())
display(report.head(20))

Total rows: 5951
Rows still containing empty cells: 6


,utc_time,beck_time,sie_time,empty_cells
0,2019-09-26 18:21:14.515+00:00,NaT,2019-09-26 18:21:14.515+00:00,56
1,2019-09-26 18:21:14.600+00:00,NaT,2019-09-26 18:21:14.515+00:00,56
2,2019-09-26 18:21:14.685+00:00,NaT,2019-09-26 18:21:14.515+00:00,56
3,2019-09-26 18:21:14.770+00:00,NaT,2019-09-26 18:21:14.515+00:00,56
4,2019-09-26 18:21:14.855+00:00,NaT,2019-09-26 18:21:14.850+00:00,56
5950,2019-09-26 18:29:40.265+00:00,2019-09-26 18:29:40.218+00:00,2019-09-26 18:29:40.239+00:00,118


**What**
- Drop the two fully empty columns `Unnamed: 54` and `Unnamed: 143`
- Build the missing-cell mask again and list the rows that still have empty cells

**Why**
- Both columns come from a trailing empty header in the CSV files
- They have no values and no dictionary entry (cell 21), so they carry no information
- It is the only column removal we make, and we drop by name, not by a rule such as "drop every empty column", so nothing else can disappear unnoticed

**Output**
- Total rows: 5951
- Rows still containing empty cells: 6
- Five are the first grid rows (18:21:14.515 to 18:21:14.855), before Beckhoff starts
- The sixth is the last grid row at 18:29:40.265

**Next**
- Decide what to do with these 6 rows
- Run this cell only once and in order, a second run fails because the columns are already gone

In [23]:
rows_before = len(grid)
grid = grid.dropna()  # drops every row with at least one NaN cell

print("Rows removed:", rows_before - len(grid))
print("Rows remaining:", len(grid))
print("Missing cells remaining:", grid.isna().sum().sum())  # expect 0

Rows removed: 6
Rows remaining: 5945
Missing cells remaining: 0


**What**
- Remove the rows that still have an empty cell (`dropna`)
- Check how many rows were removed and that no missing cell is left

**Why**
- The later analysis needs complete rows
- We remove only 6 of 5951 rows (about 0.1%) and we know where they are (cell 23): 5 at the very start of the log, before Beckhoff begins, and 1 at the end
- This is the only place where we remove rows, and it is our decision

**Output**
- Rows removed: 6
- Rows remaining: 5945
- Missing cells remaining: 0

**Next**
- Check the final table size and save it

In [24]:
print("Rows:", grid.shape[0])     # number of rows
print("Columns:", grid.shape[1])  # number of columns

Rows: 5945
Columns: 202


In [25]:
out_path = OUT_DIR / "merged_85ms.csv"

grid.to_csv(out_path, index=False)  # index=False: no row numbers in file

# Added: confirm what was saved (no change to the saved data).
print("Saved:", out_path)
print("Rows:", len(grid), "| Columns:", len(grid.columns))


Saved: /Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Generated_Outputs/merged_85ms.csv
Rows: 5945 | Columns: 202


out_path = (
    ""
    "output directory/merged_85ms.csv"
)

grid.to_csv(out_path, index=False)  # index=False: no row numbers in file

# Added: confirm what was saved (no change to the saved data).
print("Saved:", out_path)
print("Rows:", len(grid), "| Columns:", len(grid.columns))